In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("เชื่อมต่อ Drive สำเร็จ!")

In [ ]:
!pip install -q pycocotools mediapipe opencv-python-headless tqdm
import cv2, mediapipe as mp, numpy as np
print("ติดตั้งเครื่องมือ MediaPipe และ OpenCV เรียบร้อย!")

In [ ]:
import os

# กำหนด Path หลักที่เคยสร้างไว้ใน Notebook แรก
BASE = '/content/drive/MyDrive/exercise_pose'
ANNO = f'{BASE}/data/coco/annotations'
IMGS = f'{BASE}/data/coco/images'
PROC = f'{BASE}/data/processed'

print(f"ระบบพร้อมทำงานที่โฟลเดอร์: {BASE}")

In [ ]:
import numpy as np

def is_valid_sample(ann, min_kps=8, min_area=3000):
    """
    กรอง annotation ที่เหมาะกับ exercise analysis
    - num_keypoints >= 8 (keypoints visible พอ)
    - lower body visible >= 4 จุด (สะโพก เข่า ข้อเท้า)
    - area >= 3000 (คนไม่เล็กเกินจนอ่านไม่ออก)
    """
    # ย่อหน้าทุกบรรทัดด้านล่างนี้ให้เท่ากัน
    kps = np.array(ann['keypoints']).reshape(-1, 3)
    lower = kps[11:17] # hip(11,12) knee(13,14) ankle(15,16)
    lower_visible = (lower[:, 2] > 0).sum()

    return (
        ann['num_keypoints'] >= min_kps and
        lower_visible >= 4 and
        ann['area'] >= min_area
    )

print("ฟังก์ชัน is_valid_sample ถูกสร้างสำเร็จแล้ว!")

In [ ]:
from tqdm.notebook import tqdm

# Assuming 'coco' object is defined elsewhere and loaded with annotations
# For example: from pycocotools.coco import COCO; coco = COCO(annotation_file_path)
# If 'coco' is not yet defined, this will raise a NameError after the SyntaxError is fixed.

all_ann_ids = coco.getAnnIds(catIds=1, iscrowd=False)
all_anns = coco.loadAnns(all_ann_ids)

good_samples = [] # Initialize as an empty list for appending

for ann in tqdm(all_anns, desc="Filtering"):
    if is_valid_sample(ann):
        kps = np.array(ann['keypoints']).reshape(-1, 3)
        good_samples.append({
            'img_id': ann['image_id'],
            'ann_id': ann['id'], # Corrected non-standard unicode characters to 'id'
            "kps": kps,
            "bbox": ann['bbox'],
            'area': ann['area'],
        }) # Dictionary and append function are now properly closed

# Corrected print statements
print(f"จำนวนทั้งหมด: {len(all_anns):,}")
print(f"หลัง Filter: {len(good_samples):,}")
if len(all_anns) > 0:
    print(f"คิดเป็น: {len(good_samples)/len(all_anns)*100 :.1f}%")
else:
    print("ไม่มี annotation ทั้งหมดให้กรอง")

In [ ]:
#เซฟ filtered samples เป็น .pkl เซฟไว้ใน Drive เพื่อไม่ต้องรัน filter ใหม่ทุกครั้ง
import pickle, os

CACHE = f'{PROC}/samples_val_filtered.pkl'
with open(CACHE, 'wb') as f:
    pickle.dump(good_samples, f)

size = os.path.getsize(CACHE) / 1e6
print(f"บันทึกแล้ว: {CACHE}")
print(f"ขนาดไฟล์: {size :.1f} MB")
print(f"จำนวน: {len(good_samples):,} samples")